# Importing data

In [1]:
# Imports

library(data.table)
library(readxl)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   3.5.1     ✔ tibble    3.3.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   masks data.table::transpose()
✖ lubridate::wday() 

The introductory spiel of the chapter is that, so far, we have been using datasets already available as R objects exposed by libraries. Far more frequently (i.e. almost all of the time) the data will come from a plain text file, database or other source. Chapter 6 concerns loading data from external sources and in support of that goal we will first learn about the overall process of navigating the file system.

## Navigating and managing the file system

The book talks briefly about loading datasets through RStudio's (not even being used here) graphical interface but discourages it as writing code will result in an automated, reproducible process.

### The file system

Described here are the very familiar concepts of:

* Directories
* The root directory
* The working directory

### Relative and full paths

These are very familiar concepts but here's how to get the full path to an installed R package:

In [2]:
system.file(package = 'dslabs')

[1] "/home/readyready15728/R/x86_64-pc-linux-gnu-library/4.4/dslabs"

Here's how to do the equivalent of `ls` with a given directory:

In [3]:
dir <- system.file(package = 'dslabs')
list.files(dir)

[1] "data"        "DESCRIPTION" "extdata"     "help"        "html"       
 [6] "INDEX"       "Meta"        "NAMESPACE"   "R"           "script"

Clearly these are relative paths. `system.file()` really isn't used much in practice but the book mentions it anyway because directory `extdata` contains datasets that may be of interest:

In [4]:
file.path(dir, 'extdata') |> list.files()

[1] "2010_bigfive_regents.xls"                               
 [2] "calificaciones.csv"                                     
 [3] "carbon_emissions.csv"                                   
 [4] "fertility-two-countries-example.csv"                    
 [5] "HRlist2.txt"                                            
 [6] "life-expectancy-and-fertility-two-countries-example.csv"
 [7] "murders.csv"                                            
 [8] "olive.csv"                                              
 [9] "RD-Mortality-Report_2015-18-180531.pdf"                 
[10] "ssa-death-probability.csv"

### The working directory

`getwd()` and `setwd()` do exactly what they sound like. The book advises to use relative paths only in your projects as hard-coding full paths will very likely break on another user's system. (Also, I just noticed that the default working directory for consoles in JupyterLab depends on where you are in the file browser on the left-hand side.)

### Generating path names

The aforementioned function `file.path()` *portably* generates file paths (i.e. with `\` on Windows and `/` everywhere else):

In [5]:
file_path <- file.path(dir, 'extdata', 'murders.csv')
file_path

[1] "/home/readyready15728/R/x86_64-pc-linux-gnu-library/4.4/dslabs/extdata/murders.csv"

Also, `file.copy()` can be used for the obvious purpose of copying files. Here's how to copy `murders.csv` to the working directory (will be needed later):

In [6]:
file.copy(file_path, '.', overwrite = TRUE)

[1] TRUE

## File types

Broadly speaking file formats used by R and other such computing packages can be classified into text and binary. The most widely used formats and how to identify them are described here.

### Text files

The book talks about the different delimited text formats such as CSV and suggests that you look at the file yourself if unsure which one is being used.

### Binary files

The book really doesn't have much to say about actually identifying different types of binary files but of course here on Linux the `file` utility fills that niche nicely.

### Encoding

The pesky matter of encoding has now reared its ugly head! R defaults to ASCII/UTF-8 but other encodings are possible. The package dslabs includes a dataset that uses ISO-8859-1:

In [7]:
fn <- 'calificaciones.csv'
file.copy(file.path(system.file('extdata', package = 'dslabs'), fn), fn, overwrite = TRUE)
readLines('calificaciones.csv', n = 1)

[1] TRUE

[1] "\"nombre\",\"f.n.\",\"estampa\",\"puntuaci\xf3n\""

It's worth noting that `file -i` can be used to determine the encoding of a text file. In this case, the output for `calificaciones.csv` is:

```
calificaciones.csv: text/csv; charset=iso-8859-1
```

## Parsers

R includes its own built-in file reading functionality obviously but third-party libraries, such as readr, readxl and data.table are preferred.

### Base R

Among the built-in reader functions in R are `read.csv()`, `read.table()` and `read.delim()`. The first argument can be absolute or relative. If relative, R assumes you're using its current working directory:

In [8]:
dat <- read.csv('murders.csv')
head(dat)

,state,abb,region,population,total
,<chr>,<chr>,<chr>,<int>,<int>
1,Alabama,AL,South,4779736,135
2,Alaska,AK,West,710231,19
3,Arizona,AZ,West,6392017,232
4,Arkansas,AR,South,2915918,93
5,California,CA,West,37253956,1257
6,Colorado,CO,West,5029196,65


The book also very briefly describes `scan()` but I'm not entirely sure how useful it really is. The book barely motivated its use honestly.

### readr

readr is a Tidyverse package that reads different types of "rectangular" text files. `library(tidyverse)` will of course bring it in but `library(readr)` also works if you don't necessarily want all Tidyverse packages. Among the functions it includes are:

| Function         | Format                                          | Typical suffix  |
|------------------|-------------------------------------------------|-----------------|
| `read_table()`   | whitespace-separated values                     | .txt            |
| `read_csv()`     | comma-separated values                          | .csv            |
| `read_csv2()`    | semicolon-separated values                      | .csv            |
| `read_tsv()`     | tab-delimited values                            | .tsv            |
| `read_delim()`   | general text file format; must define delimiter | .txt            |

readr also has `read_lines()` which works a bit like `readLines()`. Here's an example of readr being used with `murders.csv`:

dat <- read_csv('murders.csv')
head(dat)

Note that diagnostic output on how the text columns were turned into R data types appears (can be suppressed with `show_col_type = FALSE`):

In [9]:
dat <- read_csv('murders.csv', show_col_types = FALSE)
head(dat)

state,abb,region,population,total
<chr>,<chr>,<chr>,<dbl>,<dbl>
Alabama,AL,South,4779736,135
Alaska,AK,West,710231,19
Arizona,AZ,West,6392017,232
Arkansas,AR,South,2915918,93
California,CA,West,37253956,1257
Colorado,CO,West,5029196,65


Note also that `dat` is a tibble and not just a regular data frame.

readr also permits the specification of an encoding. It also includes the function `guess_encoding()` which does exactly what it says:

In [10]:
guess_encoding('murders.csv')

encoding,confidence
<chr>,<dbl>
ASCII,1


In [11]:
guess_encoding('calificaciones.csv')

encoding,confidence
<chr>,<dbl>
ISO-8859-1,0.92
ISO-8859-2,0.72
ISO-8859-9,0.53


Once an encoding has been figured out, it can be specified using the `locale` argument and accordingly now `calificaciones.csv` can be read properly:

In [12]:
dat <- read_csv('calificaciones.csv', locale = locale(encoding = 'ISO-8859-1'))
head(dat)

Rows: 7 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): nombre, f.n.
num  (1): puntuación
dttm (1): estampa

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


nombre,f.n.,estampa,puntuación
<chr>,<chr>,<dttm>,<dbl>
Beyoncé,04 de septiembre de 1981,2023-09-22 02:11:02,875
Blümchen,20 de abril de 1980,2023-09-22 03:23:05,990
João,10 de junio de 1931,2023-09-21 22:43:28,989
López,24 de julio de 1969,2023-09-22 01:06:59,887
Ñengo,15 de diciembre de 1981,2023-09-21 23:35:37,931
Plácido,24 de enero de 1941,2023-09-21 23:17:21,887


### readxl

readxl provides functions to work with Microsoft Excel documents. The book really does not get into how they work very much at all but apparently these are some common entry points to an Excel workflow with R:

| Function       | Format                 | Typical suffix   |
|----------------|------------------------|------------------|
| `read_excel()` | auto-detect the format | .xls, .xlsx      |
| `read_xls()`   | original format        | .xls             |
| `read_xlsx()`  | new format             | .xlsx            |

I'm willing to guess that most people just use `read_excel()`. Also, of course there can be multiple spreadsheets in the same file. The book describes how to get sheets other than the default one (i.e. the first) out of an Excel file, using `excel_sheets()` to find the names of the sheets and the optional argument `sheet`, but passes up a perfectly good opportunity to *demonstrate* the use of said functionality. Here it is:

In [13]:
big_five_path <- file.path(dir, 'extdata', '2010_bigfive_regents.xls')
excel_sheets(big_five_path)

[1] "Sheet1" "Sheet2" "Sheet3"

Here's the default sheet, `Sheet1`:

In [14]:
read_excel(big_five_path) |> head()

label,INTEGRATED ALGEBRA,GLOBAL HISTORY,LIVING ENVIRONMENT,ENGLISH,U.S. HISTORY
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
test_year,2010,2010,2010,2010,2010
Scores,131024,113804,104201,103886,91756
0,56,55,66,165,65
1,NA,8,3,69,4
2,1,9,2,237,16
3,NA,3,1,190,10


Let's try getting `Sheet2`:

In [15]:
read_excel(big_five_path, sheet = 'Sheet2') |> head()

<0 x 0 matrix>

Oh crap, it's blank. (Maybe not the best pedagogy.)

### data.table

data.table offers `fread()`, which, of course, is geared towards larger datasets. It automatically detects the format and can even to a certain extent work with compressed files using formats such `.gz` and `.zip`. Of course `fread()` returns a data.table object:

In [16]:
fread('murders.csv') |> head()

state,abb,region,population,total
<chr>,<chr>,<chr>,<int>,<int>
Alabama,AL,South,4779736,135
Alaska,AK,West,710231,19
Arizona,AZ,West,6392017,232
Arkansas,AR,South,2915918,93
California,CA,West,37253956,1257
Colorado,CO,West,5029196,65


### Downloading files

R, even with base functions at least sometimes, can also work with an obscure invention called the World Wide Web. Here's how to pull `murders.csv` from the book's GitHub:

In [17]:
url <- paste0('https://raw.githubusercontent.com/',
              'rafalab/dslabs/master/inst/extdata/murders.csv')
read.csv(url) |> head()

,state,abb,region,population,total
,<chr>,<chr>,<chr>,<int>,<int>
1,Alabama,AL,South,4779736,135
2,Alaska,AK,West,710231,19
3,Arizona,AZ,West,6392017,232
4,Arkansas,AR,South,2915918,93
5,California,CA,West,37253956,1257
6,Colorado,CO,West,5029196,65


`download.file()` copies the online file to the working directory. **Note that `download.file()` will silently overwrite an existing file of the same name! You can look before you leap with `file.exists()`.**

In [18]:
download.file(url, 'murders.csv')

I'll just crib the next part from the book:

"Two functions that are sometimes useful when downloading data from the Internet are `tempdir()` and `tempfile()`. The first creates a directory with a random name that is very likely to be unique. Similarly, `tempfile()` creates a character string, not a file, that is likely to be a unique filename. So you can run a command like this which erases the temporary file once it imports the data:"

In [19]:
tmp_filename <- tempfile()
tmp_filename

[1] "/tmp/RtmpGQfeTP/file2348e9fb098a6"

In [20]:
download.file(url, tmp_filename)
dat <- read_csv(tmp_filename)
head(dat)

Rows: 51 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): state, abb, region
dbl (2): population, total

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


state,abb,region,population,total
<chr>,<chr>,<chr>,<dbl>,<dbl>
Alabama,AL,South,4779736,135
Alaska,AK,West,710231,19
Arizona,AZ,West,6392017,232
Arkansas,AR,South,2915918,93
California,CA,West,37253956,1257
Colorado,CO,West,5029196,65


In [21]:
file.remove(tmp_filename)

[1] TRUE

## Organizing data with spreadsheets

Just before the exercises, the book gives a bit of advice about how to fill out spreadsheets so that analysis will be tractable, even though its main concern is data *analysis* rather than data *management* and also the reader is encouraged to automate the creation of datasets as much as possible as opposed to filling things in all manually. Here are the guidelines:

* **Be consistent**: Before you commence entering data, have a plan. Once you have a plan, be consistent and stick to it.
* **Choose good names for things**: You want the names you pick for objects, files, and directories to be memorable, easy to spell and descriptive. This is actually a hard balance to achieve and it does require time and thought. One important rule to follow is **do not use spaces**, use underscores (`_`) or dashes (`-`) instead. Also, avoid symbols; stick to letters and numbers.
* **Write dates as `YYYY-MM-DD`**: To avoid confusion, we strongly recommend using this global ISO 8601 standard.
* **No empty cells**: Fill in all cells and use some common code for missing data.
* **Put just one thing in a cell**: It is better to add columns to store the extra information rather than having more than one piece of information in one cell.
* **Make it a rectangle**: The spreadsheet should be a rectangle.
* **Create a data dictionary**: If you need to explain things, such as what the columns are or what the labels used for categorical variables are, do this in a separate file.
* **No calculations in the raw data files**: Excel permits you to perform calculations. Do not make this part of your spreadsheet. Code for calculations should be in a script.
* **Do not use font color or highlighting as data**: Most import functions are not able to import this information. Encode this information as a variable instead.
* **Make backups**: Make regular backups of your data.
* **Use data validation to avoid errors**: Leverage the tools in your spreadsheet software so that the process is as error-free and repetitive-stress-injury-free as possible.
* **Save the data as text files**: Save files for sharing in comma or tab delimited format.

## Exercises

**Exercise 1**: Use the `read_csv()` function to read each of the files that the following code saves in the `files` object:

In [22]:
path <- system.file('extdata', package = 'dslabs')
files <- list.files(path)
files

[1] "2010_bigfive_regents.xls"                               
 [2] "calificaciones.csv"                                     
 [3] "carbon_emissions.csv"                                   
 [4] "fertility-two-countries-example.csv"                    
 [5] "HRlist2.txt"                                            
 [6] "life-expectancy-and-fertility-two-countries-example.csv"
 [7] "murders.csv"                                            
 [8] "olive.csv"                                              
 [9] "RD-Mortality-Report_2015-18-180531.pdf"                 
[10] "ssa-death-probability.csv"

I really don't feel like doing all of that. Instead I will skip to...

**Exercise 2**: Note that the the `olive.csv` file gives us a warning. This is because the first line of the file is missing the header for the first column.

In [23]:
olive_path <- file.path(path, 'olive.csv')
read_csv(olive_path) |> head()

New names:
• `` -> `...1`
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 572 Columns: 11
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): Region, eicosenoic
dbl (9): ...1, Area, palmitic, palmitoleic, stearic, oleic, linoleic, linole...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


...1,Region,Area,palmitic,palmitoleic,stearic,oleic,linoleic,linolenic,arachidic,eicosenoic
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,North-Apulia,1,1,1075,75,226,7823,672,36,"60,29"
2,North-Apulia,1,1,1088,73,224,7709,781,31,"61,29"
3,North-Apulia,1,1,911,54,246,8113,549,31,"63,29"
4,North-Apulia,1,1,966,57,240,7952,619,50,"78,35"
5,North-Apulia,1,1,1051,67,259,7771,672,50,"80,46"
6,North-Apulia,1,1,911,49,268,7924,678,51,"70,44"


Read the help file for `read_csv()` to figure out how to read in the file without reading this header. If you skip the header, you should not get this warning. Save the result to an object called `dat`.

In [24]:
dat <- read_csv(olive_path, col_names = FALSE, skip = 1)
head(dat)

Rows: 572 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): X2
dbl (11): X1, X3, X4, X5, X6, X7, X8, X9, X10, X11, X12

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,North-Apulia,1,1,1075,75,226,7823,672,36,60,29
2,North-Apulia,1,1,1088,73,224,7709,781,31,61,29
3,North-Apulia,1,1,911,54,246,8113,549,31,63,29
4,North-Apulia,1,1,966,57,240,7952,619,50,78,35
5,North-Apulia,1,1,1051,67,259,7771,672,50,80,46
6,North-Apulia,1,1,911,49,268,7924,678,51,70,44


**Exercise 3**: A problem with the previous approach is that we don't know what the columns represent. Type:

In [25]:
names(dat)

[1] "X1"  "X2"  "X3"  "X4"  "X5"  "X6"  "X7"  "X8"  "X9"  "X10" "X11" "X12"

...to see that the names are not informative. Use the `readLines()` function to read in just the first line.

In [26]:
readLines(olive_path, n = 1)

[1] ",Region,Area,palmitic,palmitoleic,stearic,oleic,linoleic,linolenic,arachidic,eicosenoic"

**Exercise 4**: Pick a measurement you can take on a regular basis. For example, your daily weight or how long it takes you to run 5 miles. Keep a spreadsheet that includes the date, the hour, the measurement, and any other informative variable you think is worth keeping. Do this for 2 weeks. Then make a plot.

(Not doing this one either, plus, what *kind* of plot?)